In [29]:
import pandas as pd
import xgboost as xgb

df = pd.read_parquet('stocks.parquet')

df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

features = df.columns.difference(['id', 'date', 'ret_eom', 'gvkey', 'iid', 'excntry',
    'year', 'month', 'char_date', 'char_eom', 'stock_ret']).tolist()

# Initial training and validation periods
initial_train_end = pd.to_datetime("2012-12-31")
validation_end    = pd.to_datetime("2014-12-31")
end_date          = pd.to_datetime("2025-05-31")

# Collect predictions
predictions = []

while validation_end < end_date:

    # Masks
    train_mask = (df["date"] <= initial_train_end)
    valid_mask = (df["date"] > initial_train_end) & (df["date"] <= validation_end)
    test_mask  = (df["date"] > validation_end) & (df["date"] <= validation_end + pd.DateOffset(years=1))

    train_X, train_y = df.loc[train_mask, features], df.loc[train_mask, "stock_ret"]
    valid_X, valid_y = df.loc[valid_mask, features], df.loc[valid_mask, "stock_ret"]
    test_X           = df.loc[test_mask, features]

    dtrain = xgb.DMatrix(train_X, label=train_y)
    dvalid = xgb.DMatrix(valid_X, label=valid_y)
    dtest  = xgb.DMatrix(test_X)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 12,
        "eta": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "seed": 42,
        "tree_method": "gpu_hist",   # 🚀 enables GPU acceleration
        "predictor": "gpu_predictor" # (optional) ensures GPU is used for prediction too
    }

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=50,
        verbose_eval=True,
    )

    # Predictions for test window
    preds = model.predict(dtest)

    # Save with stock_id + date
    tmp = df.loc[test_mask, ["gvkey", "date", "stock_ret"]].copy()
    tmp["predicted_return"] = preds
    predictions.append(tmp)

    # Expand window by 1 year
    initial_train_end += pd.DateOffset(years=1)
    validation_end += pd.DateOffset(years=1)


pred_df = pd.concat(predictions, ignore_index=True)



c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:17:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:17:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


[0]	train-rmse:0.82969	valid-rmse:0.18352
[1]	train-rmse:0.81015	valid-rmse:0.18912
[2]	train-rmse:0.79099	valid-rmse:0.19079
[3]	train-rmse:0.77902	valid-rmse:0.19294
[4]	train-rmse:0.76071	valid-rmse:0.19568
[5]	train-rmse:0.74391	valid-rmse:0.19844
[6]	train-rmse:0.72639	valid-rmse:0.20160
[7]	train-rmse:0.72553	valid-rmse:0.20175
[8]	train-rmse:0.70981	valid-rmse:0.20260
[9]	train-rmse:0.69383	valid-rmse:0.20668
[10]	train-rmse:0.69284	valid-rmse:0.20751
[11]	train-rmse:0.67691	valid-rmse:0.21001
[12]	train-rmse:0.66247	valid-rmse:0.21182
[13]	train-rmse:0.64897	valid-rmse:0.21887
[14]	train-rmse:0.63480	valid-rmse:0.23024
[15]	train-rmse:0.62154	valid-rmse:0.24409
[16]	train-rmse:0.60786	valid-rmse:0.25476
[17]	train-rmse:0.59410	valid-rmse:0.25933
[18]	train-rmse:0.58090	valid-rmse:0.26500
[19]	train-rmse:0.56876	valid-rmse:0.26688
[20]	train-rmse:0.55609	valid-rmse:0.27263
[21]	train-rmse:0.54362	valid-rmse:0.27709
[22]	train-rmse:0.53155	valid-rmse:0.28230
[23]	train-rmse:0.519

c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:17:21] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:17:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarn

[0]	train-rmse:0.79006	valid-rmse:15.10869
[1]	train-rmse:0.77175	valid-rmse:15.10221
[2]	train-rmse:0.75382	valid-rmse:15.09605
[3]	train-rmse:0.73747	valid-rmse:15.09656
[4]	train-rmse:0.72022	valid-rmse:15.09093
[5]	train-rmse:0.70450	valid-rmse:15.09236
[6]	train-rmse:0.68837	valid-rmse:15.08721
[7]	train-rmse:0.68754	valid-rmse:15.08724
[8]	train-rmse:0.67275	valid-rmse:15.08841
[9]	train-rmse:0.65810	valid-rmse:15.08377
[10]	train-rmse:0.65708	valid-rmse:15.08377
[11]	train-rmse:0.64245	valid-rmse:15.08375
[12]	train-rmse:0.62901	valid-rmse:15.08530
[13]	train-rmse:0.61637	valid-rmse:15.08811
[14]	train-rmse:0.60323	valid-rmse:15.08626
[15]	train-rmse:0.59086	valid-rmse:15.09072
[16]	train-rmse:0.57820	valid-rmse:15.08540
[17]	train-rmse:0.56586	valid-rmse:15.08025
[18]	train-rmse:0.55415	valid-rmse:15.07897
[19]	train-rmse:0.54287	valid-rmse:15.08063
[20]	train-rmse:0.53139	valid-rmse:15.07989
[21]	train-rmse:0.52017	valid-rmse:15.07684
[22]	train-rmse:0.50987	valid-rmse:15.0722

c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:17:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:17:32] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarn

...

c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:18:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:18:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarn

[0]	train-rmse:9.90107	valid-rmse:0.30936
[1]	train-rmse:9.65882	valid-rmse:0.30930
[2]	train-rmse:9.65601	valid-rmse:0.31008
[3]	train-rmse:9.41533	valid-rmse:0.31007
[4]	train-rmse:9.18425	valid-rmse:0.65932
[5]	train-rmse:8.96863	valid-rmse:0.65968
[6]	train-rmse:8.91107	valid-rmse:0.66271
[7]	train-rmse:8.68910	valid-rmse:0.66268
[8]	train-rmse:8.47214	valid-rmse:0.66274
[9]	train-rmse:8.42045	valid-rmse:0.66414
[10]	train-rmse:8.22284	valid-rmse:1.22231
[11]	train-rmse:8.17446	valid-rmse:1.22222
[12]	train-rmse:8.02129	valid-rmse:1.19381
[13]	train-rmse:7.86763	valid-rmse:1.16467
[14]	train-rmse:7.68207	valid-rmse:1.16917
[15]	train-rmse:7.49012	valid-rmse:1.17436
[16]	train-rmse:7.44912	valid-rmse:1.16049
[17]	train-rmse:7.26236	valid-rmse:1.17764
[18]	train-rmse:7.08940	valid-rmse:1.17755
[19]	train-rmse:6.95278	valid-rmse:1.16669
[20]	train-rmse:6.77870	valid-rmse:1.14236
[21]	train-rmse:6.60903	valid-rmse:1.11048
[22]	train-rmse:6.45035	valid-rmse:1.07906
[23]	train-rmse:6.408

c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:18:25] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:18:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarn

[0]	train-rmse:9.60386	valid-rmse:0.30360
[1]	train-rmse:9.36887	valid-rmse:0.94404
[2]	train-rmse:9.36613	valid-rmse:0.91242
[3]	train-rmse:9.13263	valid-rmse:0.89978
[4]	train-rmse:8.90849	valid-rmse:1.82221
[5]	train-rmse:8.69937	valid-rmse:2.60816
[6]	train-rmse:8.65465	valid-rmse:2.55537
[7]	train-rmse:8.43905	valid-rmse:2.53683
[8]	train-rmse:8.22834	valid-rmse:2.48721
[9]	train-rmse:8.17824	valid-rmse:2.48799
[10]	train-rmse:7.98641	valid-rmse:3.40204
[11]	train-rmse:7.93953	valid-rmse:3.40272
[12]	train-rmse:7.79082	valid-rmse:3.40278
[13]	train-rmse:7.64132	valid-rmse:3.39164
[14]	train-rmse:7.46131	valid-rmse:3.40473
[15]	train-rmse:7.27522	valid-rmse:3.36287
[16]	train-rmse:7.23538	valid-rmse:3.30103
[17]	train-rmse:7.05431	valid-rmse:3.25059
[18]	train-rmse:6.88669	valid-rmse:3.25117
[19]	train-rmse:6.75383	valid-rmse:3.20077
[20]	train-rmse:6.58485	valid-rmse:3.17863
[21]	train-rmse:6.42003	valid-rmse:3.46101
[22]	train-rmse:6.26598	valid-rmse:4.01421
[23]	train-rmse:6.225

c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:18:32] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


In [33]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

pred_df = pd.concat(predictions, ignore_index=True)

# Drop NaNs if any (can happen if test set empty in some window)
pred_df = pred_df.dropna(subset=["stock_ret", "predicted_return"])

# Metrics
mse = mean_squared_error(pred_df["stock_ret"], pred_df["predicted_return"])
rmse = np.sqrt(mse)
mae = mean_absolute_error(pred_df["stock_ret"], pred_df["predicted_return"])
r2  = r2_score(pred_df["stock_ret"], pred_df["predicted_return"])

print(f"Overall Performance:")
print(f"  MSE  = {mse:.6f}")
print(f"  RMSE = {rmse:.6f}")
print(f"  MAE  = {mae:.6f}")
print(f"  R²   = {r2:.6f}")


pred_df

Overall Performance:
  MSE  = 192.056116
  RMSE = 13.858431
  MAE  = 0.162543
  R²   = -0.063251


,gvkey,date,stock_ret,predicted_return
0,1096.0,2015-01-30,-0.121979,0.006722
1,1186.0,2015-01-30,0.350143,-0.017229
2,1262.0,2015-01-30,-0.093005,0.007439
3,1828.0,2015-01-30,-0.059300,0.009002
4,2055.0,2015-01-30,0.182980,0.029857
...,...,...,...,...
667270,330888.0,2025-06-30,-0.013834,0.003092
667271,343592.0,2025-06-30,-0.095628,-0.068433
667272,347085.0,2025-06-30,-0.117477,0.008709
667273,349705.0,2025-06-30,-0.030758,-0.012431
